# Student Lab Pack – Part C: Multimodal AI Integration

**Topics covered:**
- Text generation and summarization
- Image generation and editing
- Speech-to-text and text-to-speech
- Multimodal reasoning workflows

We focus on **business-relevant** use cases: report summarization, support reply generation, compliance extraction, and structured output — then brief demos of image/speech where they add enterprise value (accessibility, dashboards, help-desk assistants), rather than generic "text-to-image conversion" demos.

**What you will do in this notebook:**
1. Use the same `requests.post()` REST pattern across all modalities.
2. Build production helpers for text → text (primary), image → text, audio ↔ text.
3. Chain modalities into a multimodal reasoning workflow.

**Which cells implement each block:**

| Block | Step | Sections in this notebook | What the code does |
|-------|------|---------------------------|---------------------|
| 1 | Text in → text out | §2 Text generation and summarization | `summarize_for_executive()`, `draft_support_reply()`, `summarize_batch()`, `summarize_meeting_notes()`, `extract_compliance_points()`, `generate_structured_json()`, `classify_content()`, `multi_turn_chat()` build messages, call chat API, parse text |
| 2 | Image in → text out | §3 Image (vision) | `describe_image_for_support()`, `describe_image()`, `extract_text_from_image()`, `extract_invoice_fields()` build mixed text+image payload, call vision-capable chat API |
| 3 | Audio in → text out | §4 Speech (STT) | `speech_to_text()`, `speech_to_text_v2()`, `batch_transcribe()` post audio file to STT endpoint and return transcript |
| 4 | Text in → audio out | §4 Speech (TTS) | `text_to_speech()`, `text_to_speech_v2()` post text to TTS endpoint and write audio file |
| 5 | Full pipeline | §5 Multimodal reasoning workflow | `multimodal_assistant_pipeline()` chains STT → Chat → TTS; diagram maps to Student Lab Pack lab (voice → STT → Chat → TTS) |

## 1. Pipeline view – Where each modality fits

```
  Input (text / image / audio)  →  API (model)  →  Output (text / image / audio)
        │                              │                    │
        │  Text: user question,        │  Same REST         │  Text: summary,
        │  document, ticket            │  pattern:          │  reply, answer
        │  Image: chart, screenshot    │  POST + JSON       │  Image: chart,
        │  Audio: voice note           │  (or multipart)    │  diagram
        │                              │                    │  Audio: TTS
  ──────┴──────────────────────────────┴────────────────────┴──────────────
  Your code: build payload → requests.post() → parse response
```

The **same pattern** (request → API → response) applies to every modality; only the payload and response format change.

**Modality comparison table:**

| Modality | API endpoint | Request format | Response format | Typical enterprise use case |
|----------|-------------|----------------|-----------------|---------------------------|
| Text → Text | `/chat/completions` | JSON (messages array) | JSON (choices[0].message.content) | Report summaries, support drafts, compliance extraction |
| Image → Text | `/chat/completions` | JSON (messages with image_url) | JSON (text description) | Screenshot analysis, invoice OCR, accessibility |
| Audio → Text | `/audio/transcriptions` | Multipart (file + model) | JSON (text field) | Meeting notes, voice commands, call transcripts |
| Text → Audio | `/audio/speech` | JSON (input + voice) | Binary audio bytes | Voice assistants, accessibility, notifications |

### Production deployment view

```
  Customer Portal / Internal App
        │
        ▼
  ┌─────────────────┐     ┌──────────────┐     ┌───────────────────┐
  │  Your Python     │────▶│  API Gateway  │────▶│  AI Service       │
  │  (microservice)  │     │  (auth, rate  │     │  (OpenAI, Azure,  │
  │                  │◀────│   limiting)   │◀────│   on-prem model)  │
  └─────────────────┘     └──────────────┘     └───────────────────┘
        │
        ▼
  ┌─────────────────┐
  │  Response Cache  │  ← optional: cache identical requests to reduce cost
  └─────────────────┘
```

In production, the same functions you write here run inside a web service (FastAPI, Flask, etc.). The gateway handles auth and rate limiting; your code focuses on payload construction, response parsing, and error recovery — exactly what we practice below.

## 2. Setup — load secrets and config (run this cell first)

Same pattern as Part A: `.env` for secrets, `config.json` for non-secret settings. Run Jupyter from the `hands-on` folder so paths resolve.

In [86]:
# Run this cell first: load secrets and config (run Jupyter from hands-on folder)
import os
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")
with open(Path.cwd() / "config.json", encoding="utf-8") as f:
    CONFIG = json.load(f)
print("Config and .env loaded. API key set:", bool(os.environ.get("OPENAI_API_KEY")))

Config and .env loaded. API key set: True


In [87]:
# Environment check: ensure we have API base and model from config (same pattern as Part A)
def check_env():
    api_base = CONFIG.get("openai_api_base") or "https://api.openai.com/v1"
    model = CONFIG.get("model_chat") or "gpt-4o-mini"
    key_set = bool(os.environ.get("OPENAI_API_KEY"))
    return {"api_base": api_base, "model": model, "key_set": key_set}

env_status = check_env()
print("API base:", env_status["api_base"])
print("Chat model:", env_status["model"])
print("API key configured:", env_status["key_set"])

API base: https://api.openai.com/v1
Chat model: gpt-4o-mini
API key configured: True


## 3. Text generation and summarization (primary focus)

**Real-life scenarios (text → text):**
- **Internal reports:** Summarize long documents into executive bullet points for leadership so they don't have to read 10+ pages.
- **Support:** Generate a first-draft reply from a knowledge base and ticket content, for human agents to review and edit.
- **Compliance:** Extract key claims or actions from a policy document for a compliance checklist.
- **Content moderation:** Classify user-generated content as safe, unsafe, or needs_review with confidence.
- **Structured output:** Force the model to return JSON with specific keys (risk_level, reasons, next_steps).
- **Multi-turn conversation:** Maintain context across multiple questions for follow-up analysis.

We use the **same chat/completions API** with different system prompts and inputs for each scenario.

In [88]:
import requests

# Reusable helper: build chat payload (same REST pattern for all text scenarios)
def build_chat_payload(
    system_content: str,
    user_content: str,
    max_tokens: int = 300,
    temperature: float = 0.3,
    response_format: dict | None = None,
) -> dict:
    """Build OpenAI-style chat/completions request body.
    Real-world: one place to adjust model, defaults, or add response_format."""
    payload = {
        "model": CONFIG.get("model_chat", "gpt-4o-mini"),
        "messages": [
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_content[:8000]},
        ],
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    if response_format:
        payload["response_format"] = response_format
    return payload

print("build_chat_payload() defined. Used by every text scenario below.")

build_chat_payload() defined. Used by every text scenario below.


In [89]:
def call_chat_api(payload: dict, api_key: str = None) -> dict:
    """POST to chat/completions and return full response JSON.
    Centralizes URL construction, headers, timeout, and error raising."""
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        return {"choices": [{"message": {"content": "[Set OPENAI_API_KEY in .env to run real call]"}}]}
    base = CONFIG.get("openai_api_base", "https://api.openai.com/v1").rstrip("/")
    url = f"{base}/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    resp = requests.post(url, json=payload, headers=headers, timeout=60)
    resp.raise_for_status()
    return resp.json()

def extract_content(resp_json: dict) -> str:
    """Extract assistant message content from chat/completions response."""
    return resp_json.get("choices", [{}])[0].get("message", {}).get("content", "")

print("call_chat_api() and extract_content() defined — used by all scenarios below.")

call_chat_api() and extract_content() defined — used by all scenarios below.


### Scenario 1 — Executive summary (internal reports)

**Use case:** Leadership receives 10-page quarterly reports from every region. Your pipeline reads each report and returns 3-bullet executive summaries so the CTO can review all regions in 5 minutes.

This implements **Block 1 (text in → text out)** from the pipeline diagram above.

In [90]:
def summarize_for_executive(long_text: str, bullet_count: int = 3, api_key: str = None) -> str:
    """Produce an N-bullet executive summary. Use case: internal reports."""
    payload = build_chat_payload(
        f"You are an assistant that writes concise executive summaries. "
        f"Output exactly {bullet_count} bullet points. Each bullet starts with '- '.",
        long_text,
        max_tokens=400,
    )
    resp = call_chat_api(payload, api_key)
    return extract_content(resp)

sample_report = """
Q3 results show revenue up 12% YoY driven by enterprise contracts in APAC.
Operations expanded to two new regions (LATAM and MEA) with local offices.
Customer satisfaction scores improved from 4.1 to 4.5 out of 5.
Two product launches (Advanced Analytics and Mobile SDK) delayed to Q4 due to
dependency on the new auth platform. Engineering headcount grew by 15%.
Cloud infrastructure costs increased 8% but unit cost per customer dropped 3%.
Churn rate decreased from 5.2% to 4.1% — lowest in company history.
Three strategic partnerships signed (data providers, cloud vendor, consulting firm).
"""
result = summarize_for_executive(sample_report)
print("Executive summary (3 bullets):")
print(result)

Executive summary (3 bullets):
- Q3 revenue increased by 12% YoY, primarily due to enterprise contracts in the APAC region, while customer satisfaction scores rose from 4.1 to 4.5 out of 5.  
- Operations expanded into LATAM and MEA with new local offices, and churn rate decreased to a record low of 4.1%.  
- Two product launches were delayed to Q4 due to dependencies, but engineering headcount grew by 15%, and strategic partnerships were established with three key industry players.  


In [91]:
# Show the exact request body sent to the API (for learning/debugging)
sample_payload = build_chat_payload(
    "You are an assistant that writes concise executive summaries. Output exactly 3 bullet points.",
    sample_report[:200] + "...",
    max_tokens=400,
)
print("Request body (truncated input for display):")
print(json.dumps(sample_payload, indent=2)[:600])

Request body (truncated input for display):
{
  "model": "gpt-4o-mini",
  "messages": [
    {
      "role": "system",
      "content": "You are an assistant that writes concise executive summaries. Output exactly 3 bullet points."
    },
    {
      "role": "user",
      "content": "\nQ3 results show revenue up 12% YoY driven by enterprise contracts in APAC.\nOperations expanded to two new regions (LATAM and MEA) with local offices.\nCustomer satisfaction scores improved from 4.1 to..."
    }
  ],
  "max_tokens": 400,
  "temperature": 0.3
}


### Scenario 2 — Batch summarization (weekly regional reports)

**Use case:** Every Monday, 8 regional managers submit weekly updates. Your script summarizes all 8 in one run. If one report fails (e.g. timeout), the rest still complete — production pipelines must be resilient.

**Pattern:** Loop over texts, call API per text, collect results. `skip_failures=True` ensures partial results on error.

In [92]:
def summarize_batch(
    texts: list[str],
    bullet_count: int = 3,
    api_key: str = None,
    skip_failures: bool = True,
) -> list[str | None]:
    """Summarize multiple documents; returns list of N-bullet summaries.
    Real-world: weekly regional reports. skip_failures keeps the batch running."""
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    results: list[str | None] = []
    for i, text in enumerate(texts):
        try:
            payload = build_chat_payload(
                f"Output exactly {bullet_count} bullet points as executive summary.",
                text,
                max_tokens=300,
            )
            resp = call_chat_api(payload, api_key)
            results.append(extract_content(resp))
        except Exception as e:
            if skip_failures:
                results.append(f"[Error for item {i}: {e}]")
            else:
                raise
    return results

# Example: two short "reports" (run with API key for real batch)
batch_reports = [
    "North region: sales up 8%. One store closed for renovation. Staff training completed for 45 employees.",
    "South region: sales down 2%. New warehouse opened in Atlanta. Hiring 12 positions in progress.",
    "West region: flat revenue. Major client renewed for 3 years. Two support escalations resolved.",
]
batch_summaries = summarize_batch(batch_reports)
for i, s in enumerate(batch_summaries):
    print(f"Report {i+1} summary:\n{s}\n")

Report 1 summary:
- Sales in the North region increased by 8%, indicating strong performance despite challenges.
- One store is currently closed for renovation, which may impact short-term sales but aims to enhance long-term customer experience.
- Staff training has been successfully completed for 45 employees, improving workforce readiness and service quality.

Report 2 summary:
- Sales in the South region have decreased by 2%.
- A new warehouse has been opened in Atlanta to enhance distribution capabilities.
- The company is in the process of hiring for 12 positions to support operations.

Report 3 summary:
- Revenue in the West region remained flat, indicating stability but no growth in sales.
- A major client has renewed their contract for an additional three years, ensuring continued business.
- Two support escalations have been successfully resolved, enhancing client satisfaction and operational efficiency.



### Scenario 3 — Meeting notes → structured summary

**Use case:** After STT converts a meeting recording to text (see §4), the transcript is unstructured. Your pipeline extracts consistent sections: **Attendees, Key decisions, Action items**. Downstream tools (project management, email) consume these structured fields.

**Pattern:** Same chat API; the system prompt defines the output structure.

In [93]:
def summarize_meeting_notes(transcript: str, api_key: str = None) -> str:
    """Turn meeting transcript into structured summary.
    Sections: ## Attendees, ## Key decisions, ## Action items. Use case: post-meeting notes."""
    system = (
        "You summarize meeting transcripts. Output exactly three sections with these headers: "
        "## Attendees, ## Key decisions, ## Action items. "
        "Use bullet points under each. Be concise. Include owners and deadlines where mentioned."
    )
    payload = build_chat_payload(system, transcript, max_tokens=500)
    return extract_content(call_chat_api(payload, api_key))

sample_transcript = """
John: So we need to decide on the Q4 launch date. Sarah, can you confirm the backend is ready?
Sarah: Yes, backend will be ready by end of October. We still need design sign-off from marketing.
Mike: I'll chase the design sign-off from Lisa by Friday. Action for me.
John: Thanks. Next topic — budget approval. We got green light from finance for two extra hires.
Sarah: Great. I'll open the requisitions today. We need a senior engineer and a QA lead.
John: Last item — the customer escalation from Acme Corp. Mike, you're handling that?
Mike: Yes, I have a call with them Thursday. I'll report back by end of week.
John: Perfect. Meeting adjourned.
"""
meeting_summary = summarize_meeting_notes(sample_transcript)
print(meeting_summary)

## Attendees
- John
- Sarah
- Mike
- Lisa (mentioned)

## Key decisions
- Q4 launch date confirmed pending design sign-off.
- Budget approved for two extra hires: a senior engineer and a QA lead.

## Action items
- Mike to chase design sign-off from Lisa by Friday.
- Sarah to open requisitions for the senior engineer and QA lead today.
- Mike to report back on the Acme Corp escalation by the end of the week.


### Production pattern — Response metadata and cost tracking

**Use case:** Compliance and finance require audit logs of every API call: model used, tokens consumed, estimated cost. This helper extracts usage metadata alongside the content.

In [94]:
def parse_chat_response(resp_json: dict) -> tuple[str, dict]:
    """Extract content and metadata (usage, model, finish_reason) from chat/completions response."""
    choice = resp_json.get("choices", [{}])[0]
    message = choice.get("message", {})
    content = message.get("content") or ""
    usage = resp_json.get("usage", {})
    meta = {
        "model": resp_json.get("model"),
        "finish_reason": choice.get("finish_reason"),
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "total_tokens": usage.get("total_tokens"),
    }
    return content, meta

def estimate_cost(meta: dict, price_per_1k_prompt: float = 0.00015, price_per_1k_completion: float = 0.0006) -> float:
    """Estimate USD cost from token usage metadata. Prices default to gpt-4o-mini rates."""
    pt = meta.get("prompt_tokens") or 0
    ct = meta.get("completion_tokens") or 0
    return (pt / 1000) * price_per_1k_prompt + (ct / 1000) * price_per_1k_completion

# Example: call API and inspect usage
def summarize_with_usage(long_text: str) -> tuple[str, dict, float]:
    payload = build_chat_payload("Output exactly 3 bullet points.", long_text, max_tokens=300)
    resp = call_chat_api(payload)
    content, meta = parse_chat_response(resp)
    cost = estimate_cost(meta)
    return content, meta, cost

content, meta, cost = summarize_with_usage(sample_report[:500])
print("Content:", content[:200] if content else content)
print("Usage:", meta)
print(f"Estimated cost: ${cost:.6f}")

Content: - Q3 revenue increased by 12% year-over-year, primarily due to enterprise contracts in the APAC region.
- Customer satisfaction scores improved from 4.1 to 4.5 out of 5, while churn rate decreased fro
Usage: {'model': 'gpt-4o-mini-2024-07-18', 'finish_reason': 'stop', 'prompt_tokens': 143, 'completion_tokens': 83, 'total_tokens': 226}
Estimated cost: $0.000071


### Scenario 4 — Compliance extraction (same endpoint, different prompt)

**Use case:** Legal team receives regulatory documents. Your pipeline extracts key obligations, deadlines, and penalties into a structured checklist. Same `chat/completions` endpoint; the system prompt changes from "summarize" to "extract obligations."

**Pattern:** Demonstrates that the same API serves many use cases — prompt engineering is the differentiator.

In [ ]:
def extract_compliance_points(text: str, api_key: str = None) -> str:
    """Extract key obligations and deadlines from a policy document."""
    payload = build_chat_payload(
        "Extract key obligations, deadlines, and penalties as bullet points. "
        "Format each bullet as: '- [OBLIGATION] by [DEADLINE] (penalty: [PENALTY])'. "
        "If no deadline or penalty is stated, write 'N/A'. Be precise and concise.",
        text,
        max_tokens=400,
    )
    return extract_content(call_chat_api(payload, api_key))

policy_sample = """
All employees must complete annual security training by December 31, 2026.
Failure to comply results in account suspension. Customer data must be encrypted
at rest using AES-256. Monthly compliance reports are due by the 5th business day
of each month. Vendors with access to PII must sign a Data Processing Agreement
within 30 days of contract start or face contract termination. All critical
vulnerabilities must be patched within 72 hours of disclosure.
"""
print(extract_compliance_points(policy_sample))

### Scenario 5 — First-draft support reply (ticket handling)

**Use case:** Support agents receive 200+ tickets/day. For common issues, your pipeline generates a first draft using the ticket text plus a knowledge-base snippet. The agent reviews, edits, and sends — reducing average reply time from 8 minutes to 2 minutes.

**Human-in-the-loop:** The model drafts; a human reviews before sending. Never auto-send unreviewed AI output to customers.

In [ ]:
def draft_support_reply(
    ticket_text: str,
    knowledge_snippet: str,
    api_key: str = None,
) -> str:
    """Generate a short, professional reply using ticket + knowledge.
    Use for human-in-the-loop review. Never auto-send to customer."""
    user_content = f"Knowledge:\n{knowledge_snippet}\n\nTicket:\n{ticket_text}"
    payload = build_chat_payload(
        "You write brief, professional support replies. "
        "Use only the provided knowledge. Be concise. "
        "If the knowledge does not cover the issue, say so and suggest contacting support.",
        user_content,
        max_tokens=200,
    )
    return extract_content(call_chat_api(payload, api_key))

ticket = "Customer cannot reset password; getting error 500 every time they click the link."
knowledge = (
    "Password reset: use the Forgot Password link on the login page. "
    "If error 500, clear browser cache or try incognito mode. "
    "If the issue persists, it may be a backend issue — escalate to engineering with the request ID."
)
print("Draft reply:")
print(draft_support_reply(ticket, knowledge))

### Scenario 5b — Configurable tone and length

**Use case:** Some customers expect formal language (enterprise contracts); others prefer friendly, casual tone (consumer app). Same function; the system prompt adapts.

In [ ]:
def draft_support_reply_v2(
    ticket_text: str,
    knowledge_snippet: str,
    tone: str = "professional",
    max_words: int = 80,
    api_key: str = None,
) -> str:
    """First-draft support reply with configurable tone and word limit.
    Human-in-the-loop: review before sending."""
    tone_map = {
        "formal": "formal and concise",
        "friendly": "friendly and helpful",
        "professional": "professional and clear",
    }
    tone_instruction = tone_map.get(tone, "professional")
    system = (
        f"You write support replies. Use only the provided knowledge. "
        f"Tone: {tone_instruction}. Keep the reply under {max_words} words. Be actionable."
    )
    user_content = f"Knowledge:\n{knowledge_snippet}\n\nTicket:\n{ticket_text}"
    payload = build_chat_payload(system, user_content, max_tokens=250)
    return extract_content(call_chat_api(payload, api_key))

# Same ticket/knowledge, different tone
reply_friendly = draft_support_reply_v2(ticket, knowledge, tone="friendly", max_words=60)
reply_formal = draft_support_reply_v2(ticket, knowledge, tone="formal", max_words=60)
print("Friendly draft:", reply_friendly[:200])
print()
print("Formal draft:", reply_formal[:200])

### Scenario 6 — Structured JSON output (risk classification)

**Use case:** Compliance pipeline classifies incoming reports into risk levels with reasoning. Downstream systems (dashboards, alerting) consume structured JSON — not free text. The API's `response_format` parameter forces valid JSON output.

**Production pattern:** When you need machine-readable output, use `response_format: {"type": "json_object"}` and include "respond in JSON" in the system prompt. This eliminates fragile regex parsing of free-text responses.

In [ ]:
def classify_risk_as_json(text: str, api_key: str = None) -> dict:
    """Classify text into risk_level (low/medium/high/critical) with structured JSON output.
    Returns parsed dict with keys: risk_level, reasons, recommended_action."""
    system = (
        "You are a risk classification assistant. Analyze the text and respond ONLY in JSON "
        "with exactly these keys: "
        '{"risk_level": "low|medium|high|critical", '
        '"reasons": ["reason1", "reason2"], '
        '"recommended_action": "one sentence"}. '
        "No other text outside the JSON object."
    )
    payload = build_chat_payload(
        system, text, max_tokens=300, temperature=0.1,
        response_format={"type": "json_object"},
    )
    raw = extract_content(call_chat_api(payload, api_key))
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"raw_response": raw, "parse_error": True}

# Example: risk classification
incident_report = (
    "Production database experienced 45 minutes of downtime during peak hours. "
    "Approximately 12,000 customers were affected. Root cause: expired TLS certificate "
    "on the primary database connection. Failover did not trigger due to misconfigured "
    "health checks. Data integrity was maintained; no data loss confirmed."
)
risk_result = classify_risk_as_json(incident_report)
print("Risk classification (JSON):")
print(json.dumps(risk_result, indent=2))

### Scenario 7 — Content moderation / classification

**Use case:** User-generated content (reviews, forum posts, support messages) must be screened before publishing. Categories: `safe`, `needs_review`, `unsafe`. Confidence score helps prioritize human review queue.

In [ ]:
def classify_content(text: str, api_key: str = None) -> dict:
    """Classify user-generated content for moderation.
    Returns: {category: safe|needs_review|unsafe, confidence: 0.0-1.0, reason: str}"""
    system = (
        "You are a content moderation assistant. Classify the text into one of: "
        "safe, needs_review, unsafe. Respond ONLY in JSON: "
        '{"category": "safe|needs_review|unsafe", "confidence": 0.0-1.0, "reason": "brief reason"}. '
        "No text outside the JSON."
    )
    payload = build_chat_payload(
        system, text, max_tokens=150, temperature=0.1,
        response_format={"type": "json_object"},
    )
    raw = extract_content(call_chat_api(payload, api_key))
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"raw_response": raw, "parse_error": True}

# Examples: safe vs needs_review
safe_text = "Great product! The shipping was fast and the quality exceeded expectations."
flagged_text = "This is garbage. I want my money back or I will report you everywhere."
print("Safe text:", json.dumps(classify_content(safe_text), indent=2))
print()
print("Flagged text:", json.dumps(classify_content(flagged_text), indent=2))

### Scenario 8 — Multi-turn conversation (follow-up questions)

**Use case:** An analyst asks "Summarize Q3 results" then follows up with "Now compare to Q2" or "What was the churn rate?" The pipeline maintains message history so the model has context from prior turns.

**Pattern:** Append each user message and assistant reply to a `messages` list. Send the full list on each call. This is how chat UIs and stateful assistants work.

In [ ]:
def multi_turn_chat(
    messages: list[dict],
    user_text: str,
    system_prompt: str = "You are a helpful business analyst assistant.",
    api_key: str = None,
) -> tuple[str, list[dict]]:
    """Send a multi-turn conversation. Appends user message, calls API, appends assistant reply.
    Returns (assistant_text, updated_messages).
    """
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    # Ensure system prompt is first message
    if not messages or messages[0].get("role") != "system":
        messages = [{"role": "system", "content": system_prompt}] + messages
    messages.append({"role": "user", "content": user_text})
    payload = {
        "model": CONFIG.get("model_chat", "gpt-4o-mini"),
        "messages": messages,
        "max_tokens": 400,
        "temperature": 0.3,
    }
    resp = call_chat_api(payload, api_key)
    assistant_text = extract_content(resp)
    messages.append({"role": "assistant", "content": assistant_text})
    return assistant_text, messages

# Example: two-turn conversation
history: list[dict] = []
answer1, history = multi_turn_chat(history, "Summarize: revenue up 12%, two new regions, churn down to 4.1%.")
print("Turn 1:", answer1[:200])
print()
answer2, history = multi_turn_chat(history, "What was the churn rate you mentioned?")
print("Turn 2:", answer2[:200])
print(f"\nMessage history length: {len(history)} messages")

### Scenario 9 — Chain-of-thought reasoning

**Use case:** For complex decisions (e.g. "Should we approve this vendor?"), you want the model to reason step-by-step before giving a final answer. The system prompt instructs "think step by step, then state your final recommendation."

**Pattern:** The final answer is more accurate when the model explains its reasoning first. You can parse the final line or a JSON block from the response.

In [ ]:
def reason_and_decide(question: str, context: str, api_key: str = None) -> str:
    """Ask the model to think step-by-step, then give a final recommendation."""
    system = (
        "You are a senior business analyst. When given a question and context, "
        "think step by step (label each step as 'Step 1:', 'Step 2:', etc.), "
        "then state your final recommendation on a new line starting with 'RECOMMENDATION:'. "
        "Be concise but thorough."
    )
    user_content = f"Context:\n{context}\n\nQuestion: {question}"
    payload = build_chat_payload(system, user_content, max_tokens=500, temperature=0.2)
    return extract_content(call_chat_api(payload, api_key))

vendor_context = (
    "Vendor: DataPipe Inc. Annual cost: $120,000. Contract: 2 years. "
    "SOC2 Type II certified. 99.9% uptime SLA. References from 3 Fortune 500 clients. "
    "However, they were involved in a minor data incident 18 months ago (resolved, no customer impact). "
    "Alternative vendor: StreamFlow ($95,000/yr, SOC2 Type I only, 99.5% uptime, no incidents)."
)
analysis = reason_and_decide("Should we approve DataPipe Inc. as our data integration vendor?", vendor_context)
print(analysis)

### Production pattern — Prompt versioning

**Use case:** In production, prompts evolve as you iterate on quality. Storing prompts as named constants (not inline strings) enables versioning, A/B testing, and audit trails.

**Pattern:** Define prompts in one place; reference by name. When you change a prompt, the old version stays in version control.

In [ ]:
# Production pattern: prompt registry (named constants, not inline strings)
PROMPTS = {
    "executive_summary_v1": (
        "You are an assistant that writes concise executive summaries. "
        "Output exactly 3 bullet points. Each bullet starts with '- '."
    ),
    "executive_summary_v2": (
        "You write executive summaries for C-level leaders. "
        "Output exactly 3 bullet points with quantitative data where available. "
        "Each bullet starts with '- ' and is one sentence."
    ),
    "support_reply_v1": (
        "You write brief, professional support replies. "
        "Use only the provided knowledge. Be concise."
    ),
    "compliance_extract_v1": (
        "Extract key obligations, deadlines, and penalties as bullet points. "
        "Format: '- [OBLIGATION] by [DEADLINE] (penalty: [PENALTY])'. "
        "Write 'N/A' if no deadline or penalty is stated."
    ),
}

def summarize_with_prompt_version(text: str, prompt_key: str = "executive_summary_v1") -> str:
    """Summarize using a named prompt version (for A/B testing, audit)."""
    system = PROMPTS.get(prompt_key)
    if not system:
        return f"[Unknown prompt key: {prompt_key}]"
    payload = build_chat_payload(system, text, max_tokens=400)
    return extract_content(call_chat_api(payload))

# Compare v1 vs v2
print("=== v1 ===")
print(summarize_with_prompt_version(sample_report, "executive_summary_v1"))
print("\n=== v2 ===")
print(summarize_with_prompt_version(sample_report, "executive_summary_v2"))

### Production pattern — Output post-processing

**Use case:** Raw model output may contain markdown formatting, extra whitespace, or exceed length constraints. Post-processing normalizes output for downstream consumers (APIs, databases, UIs).

In [ ]:
import re

def clean_model_output(text: str, max_length: int | None = None, strip_markdown: bool = False) -> str:
    """Post-process model output: normalize whitespace, optionally strip markdown, enforce length."""
    # Normalize whitespace
    text = text.strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Optionally strip markdown bold/italic markers
    if strip_markdown:
        text = re.sub(r"\*{1,3}(.+?)\*{1,3}", r"\1", text)
        text = re.sub(r"_{1,3}(.+?)_{1,3}", r"\1", text)
        text = re.sub(r"#+\s*", "", text)
    # Enforce max length (truncate at last sentence boundary)
    if max_length and len(text) > max_length:
        truncated = text[:max_length]
        last_period = truncated.rfind(".")
        if last_period > max_length // 2:
            text = truncated[: last_period + 1]
        else:
            text = truncated.rstrip() + "..."
    return text

# Example
raw_output = "  ## Summary\n\n**Revenue** increased by *12%*.\n\n\n\nTwo regions added.  "
print("Raw:", repr(raw_output))
print("Cleaned:", repr(clean_model_output(raw_output, max_length=60, strip_markdown=True)))

### Production pattern — Model fallback chain

**Use case:** Primary model is GPT-4o-mini; if it's unavailable (rate limit, outage), fall back to GPT-3.5-turbo. In critical pipelines, always have a fallback — downtime costs more than using a slightly less capable model.

In [ ]:
def call_with_fallback(
    system: str,
    user_text: str,
    models: list[str] | None = None,
    api_key: str = None,
) -> tuple[str, str]:
    """Try primary model; on failure, try fallback(s). Returns (content, model_used)."""
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        return "[Set OPENAI_API_KEY]", "none"
    models = models or [CONFIG.get("model_chat", "gpt-4o-mini"), "gpt-3.5-turbo"]
    base = CONFIG.get("openai_api_base", "https://api.openai.com/v1").rstrip("/")
    url = f"{base}/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    last_error = None
    for model in models:
        try:
            body = {
                "model": model,
                "messages": [
                    {"role": "system", "content": system},
                    {"role": "user", "content": user_text[:8000]},
                ],
                "max_tokens": 300,
            }
            resp = requests.post(url, json=body, headers=headers, timeout=60)
            resp.raise_for_status()
            content = resp.json()["choices"][0]["message"]["content"]
            return content, model
        except Exception as e:
            last_error = e
            continue
    return f"[All models failed. Last error: {last_error}]", "none"

content, model_used = call_with_fallback("Summarize in 2 bullet points.", sample_report)
print(f"Model used: {model_used}")
print(f"Content: {content[:200]}")

### Production pattern — Token counting before sending

**Use case:** Before sending a large document to the API, estimate token count to avoid exceeding the model's context window (e.g. 128k for GPT-4o-mini). A rough rule: ~4 characters per token for English text.

**Pattern:** Pre-check avoids 400 errors and wasted API calls.

In [ ]:
def estimate_tokens(text: str, chars_per_token: float = 4.0) -> int:
    """Rough token estimate. ~4 chars/token for English. Use tiktoken for exact counts."""
    return int(len(text) / chars_per_token)

def check_and_summarize(text: str, max_input_tokens: int = 6000) -> str:
    """Check token count before sending. Truncate if too long; warn in output."""
    est = estimate_tokens(text)
    if est > max_input_tokens:
        # Truncate to fit (rough: max_input_tokens * chars_per_token)
        max_chars = int(max_input_tokens * 4)
        text = text[:max_chars]
        print(f"Warning: input truncated from ~{est} to ~{max_input_tokens} estimated tokens")
    payload = build_chat_payload(
        "Output exactly 3 bullet points as executive summary.",
        text,
        max_tokens=300,
    )
    return extract_content(call_chat_api(payload))

# Example with a short text (no truncation needed)
print(f"Sample report: ~{estimate_tokens(sample_report)} tokens")
print(check_and_summarize(sample_report))

### Smoke test — Run all text scenarios in sequence

**Use case:** Before deploying, run all text functions once to verify they work with the current API key and config. This is a quick regression check.

In [ ]:
def run_all_text_examples():
    """Smoke test all text scenarios. Prints results; use as regression check."""
    out = []
    out.append("--- Executive summary ---")
    out.append(summarize_for_executive(sample_report))
    out.append("\n--- Compliance extraction ---")
    out.append(extract_compliance_points(policy_sample))
    out.append("\n--- Support draft (professional) ---")
    out.append(draft_support_reply(ticket, knowledge))
    out.append("\n--- Risk classification (JSON) ---")
    out.append(json.dumps(classify_risk_as_json(incident_report), indent=2))
    out.append("\n--- Content moderation ---")
    out.append(json.dumps(classify_content(safe_text), indent=2))
    return "\n".join(out)

print(run_all_text_examples())

## 4. Image and vision APIs (focused use — enterprise scenarios only)

**Relevant scenarios (not generic "text-to-image"):**
- **Support screenshot analysis:** Describe an error screenshot so support agents understand the issue without opening the app.
- **Accessibility:** Generate alt-text descriptions for charts, diagrams, and UI screenshots.
- **Invoice/receipt OCR:** Extract structured data (date, amount, vendor) from scanned documents.
- **Chart interpretation:** Describe what a chart shows for automated reporting or accessibility.

We keep this section focused on **image → text** (vision) because that has the highest enterprise value. The same REST pattern applies: mixed text+image payload in the `messages` array.

This section implements **Block 2 (image in → text out)** from the pipeline diagram above.

In [ ]:
# Block 2 implementation: Vision API – describe an image (e.g. support screenshot)
def describe_image_for_support(image_url_or_base64: str, api_key: str = None) -> dict | None:
    """Send image to vision model; get text description.
    Use case: support ticket with screenshot — describe what the screenshot shows.
    """
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        return None
    base = CONFIG.get("openai_api_base", "https://api.openai.com/v1").rstrip("/")
    url = f"{base}/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    # Vision: message content is a list with text + image_url items
    body = {
        "model": CONFIG.get("model_chat", "gpt-4o-mini"),
        "messages": [
            {"role": "user", "content": [
                {"type": "text", "text": "Describe what you see in one sentence (e.g. error message, UI state)."},
                {"type": "image_url", "image_url": {"url": image_url_or_base64}},
            ]}
        ],
        "max_tokens": 150,
    }
    resp = requests.post(url, json=body, headers=headers, timeout=60)
    resp.raise_for_status()
    return resp.json()

print("Vision API: same POST + JSON pattern; content includes image_url.")
print("Request schema: messages[0].content = [{type: text}, {type: image_url}]")

### Real-life: Local image as base64

APIs accept either a public URL or a base64-encoded image (e.g. screenshot saved locally). The helper below reads a local file, encodes it, and creates a `data:image/...;base64,...` URL that the same vision endpoint accepts.

In [ ]:
import base64

def image_file_to_data_url(file_path: str, mime: str = "image/png") -> str | None:
    """Read local image file and return data URL for vision API.
    Use case: support ticket with uploaded screenshot."""
    path = Path(file_path)
    if not path.exists():
        return None
    raw = path.read_bytes()
    b64 = base64.standard_b64encode(raw).decode("ascii")
    return f"data:{mime};base64,{b64}"

def describe_image_from_file(
    file_path: str,
    prompt: str = "Describe what you see in one sentence.",
    api_key: str = None,
) -> str | None:
    """Describe a local image file (PNG/JPEG). Uses base64; same vision API as URL."""
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        return None
    data_url = image_file_to_data_url(file_path)
    if not data_url:
        return None
    data = describe_image_for_support(data_url, api_key=api_key)
    if not data or "choices" not in data:
        return None
    return data["choices"][0]["message"]["content"]

# Example: with a local file, run describe_image_from_file("screenshot.png")
print("Helpers defined: image_file_to_data_url(path), describe_image_from_file(path)")

### Brief vs detailed description

Support needs a one-line caption; accessibility needs a fuller description. Same API, different prompt and `max_tokens`.

In [ ]:
def describe_image(
    image_url_or_base64: str,
    detail: str = "brief",
    api_key: str = None,
) -> str | None:
    """Describe image: 'brief' = one sentence; 'detailed' = 2-3 sentences for accessibility."""
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        return None
    base = CONFIG.get("openai_api_base", "https://api.openai.com/v1").rstrip("/")
    url = f"{base}/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    if detail == "brief":
        text_prompt = "Describe what you see in one sentence (e.g. error message, UI state)."
        max_tokens = 150
    else:
        text_prompt = (
            "Describe the image in 2-3 sentences: what is shown, any visible text, "
            "and context (e.g. for accessibility). Be specific about UI elements, colors, and layout."
        )
        max_tokens = 300
    body = {
        "model": CONFIG.get("model_chat", "gpt-4o-mini"),
        "messages": [
            {"role": "user", "content": [
                {"type": "text", "text": text_prompt},
                {"type": "image_url", "image_url": {"url": image_url_or_base64}},
            ]}
        ],
        "max_tokens": max_tokens,
    }
    resp = requests.post(url, json=body, headers=headers, timeout=60)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

print("describe_image(url, detail='brief'|'detailed') defined.")

### Real-life: Extract text from image (OCR-style)

**Use case:** Forms, receipts, and screenshots contain text that must be captured for downstream processing. Vision models can extract and structure this text. Same endpoint, different prompt.

In [ ]:
def extract_text_from_image(image_url_or_base64: str, api_key: str = None) -> str | None:
    """Extract all visible text from image (OCR-style).
    Use case: forms, receipts, screenshots."""
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        return None
    base = CONFIG.get("openai_api_base", "https://api.openai.com/v1").rstrip("/")
    url = f"{base}/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    body = {
        "model": CONFIG.get("model_chat", "gpt-4o-mini"),
        "messages": [
            {"role": "user", "content": [
                {"type": "text", "text": (
                    "List or transcribe all text visible in this image. "
                    "Preserve structure (e.g. labels and values) if relevant."
                )},
                {"type": "image_url", "image_url": {"url": image_url_or_base64}},
            ]}
        ],
        "max_tokens": 500,
    }
    resp = requests.post(url, json=body, headers=headers, timeout=60)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

print("extract_text_from_image(url) defined. Same POST + JSON pattern; prompt targets text extraction.")

### Scenario — Invoice/receipt field extraction

**Use case:** Accounting receives scanned invoices. Your pipeline extracts key fields (vendor, date, amount, invoice number) as structured JSON for the ERP system. Vision model + JSON output format = zero manual data entry.

In [ ]:
def extract_invoice_fields(image_url_or_base64: str, api_key: str = None) -> dict:
    """Extract invoice fields from image as structured JSON.
    Fields: vendor, date, invoice_number, total_amount, currency, line_items."""
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        return {"error": "No API key"}
    base = CONFIG.get("openai_api_base", "https://api.openai.com/v1").rstrip("/")
    url = f"{base}/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    body = {
        "model": CONFIG.get("model_chat", "gpt-4o-mini"),
        "messages": [
            {"role": "system", "content": (
                "You extract invoice data from images. Respond ONLY in JSON with keys: "
                "vendor, date, invoice_number, total_amount, currency, line_items (array of "
                "{description, quantity, unit_price, amount}). Use null for missing fields."
            )},
            {"role": "user", "content": [
                {"type": "text", "text": "Extract all invoice fields from this image."},
                {"type": "image_url", "image_url": {"url": image_url_or_base64}},
            ]},
        ],
        "max_tokens": 500,
        "response_format": {"type": "json_object"},
    }
    resp = requests.post(url, json=body, headers=headers, timeout=60)
    resp.raise_for_status()
    raw = resp.json()["choices"][0]["message"]["content"]
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"raw_response": raw}

print("extract_invoice_fields(image_url) defined — returns structured JSON from invoice image.")
print("Same vision API + response_format: json_object for machine-readable output.")

### Scenario — Combined image + text diagnosis

**Use case:** A support ticket includes both a screenshot AND a text description of the problem. Your pipeline sends both to the vision model for a more accurate diagnosis.

**Pattern:** The `messages[0].content` list can include multiple text and image items.

In [ ]:
def diagnose_with_image_and_text(
    image_url_or_base64: str,
    user_description: str,
    api_key: str = None,
) -> str | None:
    """Diagnose an issue using both screenshot and user description.
    Use case: support ticket with screenshot + problem description."""
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        return None
    base = CONFIG.get("openai_api_base", "https://api.openai.com/v1").rstrip("/")
    url = f"{base}/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    body = {
        "model": CONFIG.get("model_chat", "gpt-4o-mini"),
        "messages": [
            {"role": "system", "content": (
                "You are a technical support specialist. The user provides a screenshot "
                "and a description of their problem. Analyze both, identify the likely issue, "
                "and suggest a resolution. Be concise and actionable."
            )},
            {"role": "user", "content": [
                {"type": "text", "text": f"Problem description: {user_description}"},
                {"type": "image_url", "image_url": {"url": image_url_or_base64}},
            ]},
        ],
        "max_tokens": 300,
    }
    resp = requests.post(url, json=body, headers=headers, timeout=60)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

print("diagnose_with_image_and_text(image_url, description) defined.")
print("Combines text + image in a single API call for richer context.")

## 5. Speech-to-text and text-to-speech

**Relevant scenarios:**
- **Voice input for assistants:** User speaks → STT API → text → your AI pipeline → TTS → audio reply (accessibility, hands-free).
- **Meeting notes:** Upload audio → STT → summary (combines with text summarization above).
- **Call transcript analysis:** Record customer calls → STT → sentiment analysis + action extraction.
- **Audio notifications:** Text alerts → TTS → audio file for phone/IVR systems.

Conceptually: one API call with audio bytes → text; another with text → audio bytes. Same request/response handling as text; only the Content-Type and response format change.

This section implements **Block 3 (audio in → text out)** and **Block 4 (text in → audio out)** from the pipeline diagram above.

In [ ]:
# Block 3: STT – POST audio file to transcription endpoint
def speech_to_text(audio_path: str, api_key: str = None) -> str | None:
    """Upload audio file; get transcript. Real-world: voice notes, meeting recordings.
    POST multipart form data (not JSON). Model: whisper-1 or config override."""
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        return None
    base = CONFIG.get("openai_api_base", "https://api.openai.com/v1").rstrip("/")
    url = f"{base}/audio/transcriptions"
    headers = {"Authorization": f"Bearer {api_key}"}
    with open(audio_path, "rb") as f:
        files = {"file": ("audio.mp3", f, "audio/mpeg")}
        data = {"model": CONFIG.get("model_stt", "whisper-1")}
        resp = requests.post(url, files=files, data=data, headers=headers, timeout=120)
    resp.raise_for_status()
    return resp.json().get("text")

print("STT: POST multipart file → JSON with text field.")
print("Supported formats: mp3, mp4, mpeg, mpga, m4a, wav, webm (max 25MB).")

In [ ]:
# Block 4: TTS – POST text, get audio bytes
def text_to_speech(text: str, output_path: str, api_key: str = None) -> bool:
    """Request TTS; save audio file. Use case: voice reply for assistant, accessibility.
    POST JSON body; response is binary audio (not JSON)."""
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        return False
    base = CONFIG.get("openai_api_base", "https://api.openai.com/v1").rstrip("/")
    url = f"{base}/audio/speech"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    body = {"model": CONFIG.get("model_tts", "tts-1"), "input": text, "voice": "alloy"}
    resp = requests.post(url, json=body, headers=headers, timeout=60)
    resp.raise_for_status()
    with open(output_path, "wb") as f:
        f.write(resp.content)
    return True

print("TTS: POST JSON with text → response is binary audio. Save with open(path, 'wb').")

### Real-life: STT with language hint and response format

For non-English recordings or when you need word-level timestamps, pass `language` and `response_format` (e.g. `verbose_json`) in the form data. Same multipart POST; different parameters.

In [ ]:
def speech_to_text_v2(
    audio_path: str,
    language: str | None = None,
    response_format: str = "json",
    api_key: str = None,
) -> str | dict | None:
    """STT with optional language hint and response_format.
    response_format: 'json' = text only; 'verbose_json' = text + segments with timestamps."""
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        return None
    base = CONFIG.get("openai_api_base", "https://api.openai.com/v1").rstrip("/")
    url = f"{base}/audio/transcriptions"
    headers = {"Authorization": f"Bearer {api_key}"}
    form_data = {
        "model": CONFIG.get("model_stt", "whisper-1"),
        "response_format": response_format,
    }
    if language:
        form_data["language"] = language
    with open(audio_path, "rb") as f:
        files = {"file": (Path(audio_path).name, f, "audio/mpeg")}
        resp = requests.post(url, files=files, data=form_data, headers=headers, timeout=120)
    resp.raise_for_status()
    out = resp.json()
    if response_format == "verbose_json":
        return out  # includes segments, duration, language
    return out.get("text", "")

print("speech_to_text_v2(path, language='en', response_format='verbose_json') defined.")
print("verbose_json returns segments with start/end timestamps — useful for meeting minutes.")

### Real-life: TTS with voice selection and speed

**Use case:** Different voices for different contexts (formal reports vs friendly notifications). Speed control for accessibility (slower pace for elderly users or non-native listeners).

In [ ]:
def text_to_speech_v2(
    text: str,
    output_path: str,
    voice: str = "alloy",
    speed: float = 1.0,
    api_key: str = None,
) -> bool:
    """TTS with configurable voice and speed.
    Voices: alloy, echo, fable, onyx, nova, shimmer. Speed: 0.25–4.0."""
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        return False
    base = CONFIG.get("openai_api_base", "https://api.openai.com/v1").rstrip("/")
    url = f"{base}/audio/speech"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    body = {
        "model": CONFIG.get("model_tts", "tts-1"),
        "input": text,
        "voice": voice,
        "speed": speed,
    }
    resp = requests.post(url, json=body, headers=headers, timeout=60)
    resp.raise_for_status()
    Path(output_path).write_bytes(resp.content)
    return True

TTS_VOICES = ["alloy", "echo", "fable", "onyx", "nova", "shimmer"]
print("text_to_speech_v2(text, path, voice=..., speed=...) defined.")
print("Available voices:", TTS_VOICES)

### Production pattern — Audio format detection

**Use case:** Users upload audio in various formats (mp3, wav, m4a, ogg). Your pipeline must detect the MIME type to set the correct `Content-Type` in the multipart upload.

In [ ]:
AUDIO_MIME_MAP = {
    ".mp3": "audio/mpeg",
    ".mp4": "audio/mp4",
    ".mpeg": "audio/mpeg",
    ".mpga": "audio/mpeg",
    ".m4a": "audio/mp4",
    ".wav": "audio/wav",
    ".webm": "audio/webm",
    ".ogg": "audio/ogg",
}

def detect_audio_mime(file_path: str) -> str:
    """Detect MIME type from file extension. Returns 'audio/mpeg' as default."""
    ext = Path(file_path).suffix.lower()
    return AUDIO_MIME_MAP.get(ext, "audio/mpeg")

def validate_audio_file(file_path: str, max_size_mb: int = 25) -> tuple[bool, str]:
    """Validate audio file before upload: exists, size within limit, supported format."""
    path = Path(file_path)
    if not path.exists():
        return False, f"File not found: {file_path}"
    size_mb = path.stat().st_size / (1024 * 1024)
    if size_mb > max_size_mb:
        return False, f"File too large: {size_mb:.1f} MB (max {max_size_mb} MB)"
    ext = path.suffix.lower()
    if ext not in AUDIO_MIME_MAP:
        return False, f"Unsupported format: {ext}. Supported: {list(AUDIO_MIME_MAP.keys())}"
    return True, f"OK: {path.name} ({size_mb:.1f} MB, {detect_audio_mime(file_path)})"

# Example validation
print(validate_audio_file("meeting.mp3"))  # File not found (expected)
print(detect_audio_mime("recording.wav"))   # audio/wav

### Production pattern — Batch transcription

**Use case:** After a conference, transcribe all session recordings (10–20 files). Process each file, collect results with metadata, output to CSV for the content team.

In [ ]:
import csv
from io import StringIO

def batch_transcribe(
    audio_paths: list[str],
    api_key: str = None,
    skip_failures: bool = True,
) -> list[dict]:
    """Transcribe multiple audio files. Returns list of {path, transcript, error, duration_s}."""
    import time
    results = []
    for path in audio_paths:
        valid, msg = validate_audio_file(path)
        if not valid:
            results.append({"path": path, "transcript": None, "error": msg, "duration_s": 0})
            continue
        start = time.time()
        try:
            transcript = speech_to_text(path, api_key)
            elapsed = time.time() - start
            results.append({"path": path, "transcript": transcript, "error": None, "duration_s": round(elapsed, 2)})
        except Exception as e:
            elapsed = time.time() - start
            if skip_failures:
                results.append({"path": path, "transcript": None, "error": str(e), "duration_s": round(elapsed, 2)})
            else:
                raise
    return results

def transcripts_to_csv(results: list[dict]) -> str:
    """Convert batch transcription results to CSV string."""
    output = StringIO()
    writer = csv.DictWriter(output, fieldnames=["path", "transcript", "error", "duration_s"])
    writer.writeheader()
    writer.writerows(results)
    return output.getvalue()

# Example: batch transcribe (files don't exist — shows error handling)
demo_results = batch_transcribe(["meeting1.mp3", "meeting2.wav", "missing.xyz"])
for r in demo_results:
    status = "OK" if r["transcript"] else f"FAIL: {r['error']}"
    print(f"  {r['path']}: {status}")

### Real-life: Voice note → structured summary (STT + text)

**Use case:** Manager records a 2-minute voice memo after a client call. Pipeline: STT (Block 3) → structured meeting summary (Block 1). No TTS needed — just text output for the CRM.

**Pattern:** Chain two API calls: audio → text → structured text. Same functions from above, composed together.

In [ ]:
def voice_note_to_summary(audio_path: str, api_key: str = None) -> tuple[str | None, str | None]:
    """Chain STT → structured meeting summary.
    Returns (transcript, summary). Use case: quick voice memo summary for CRM."""
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if not api_key:
        return None, None
    transcript = speech_to_text(audio_path, api_key=api_key)
    if not transcript:
        return None, None
    summary = summarize_meeting_notes(transcript, api_key=api_key)
    return transcript, summary

# Example: with a real audio file:
# transcript, summary = voice_note_to_summary("client_call_notes.mp3")
# print("Transcript:", transcript[:200]); print("Summary:", summary)
print("voice_note_to_summary(audio_path) chains STT + summarize_meeting_notes.")
print("Run with a real .mp3/.wav file when API key is set.")

## 6. Prompt engineering patterns (production techniques)

**Context:** The *same model and endpoint* produces very different outputs depending on the system prompt. This section shows three patterns that improve output quality and consistency in production.

These patterns apply to every scenario in §3 — they are a cross-cutting production skill, not a new modality.

### Pattern 1 — Same input, different system prompts → different outputs

**Use case:** You need a summary for three audiences: executive (3 bullets), engineering (technical detail), and customer communication (friendly, non-technical). One API, three prompts.

In [ ]:
def summarize_for_audience(text: str, audience: str = "executive", api_key: str = None) -> str:
    """Summarize with audience-specific system prompt."""
    audience_prompts = {
        "executive": (
            "Summarize for a C-level executive. 3 bullet points. "
            "Focus on revenue impact, risk, and strategic decisions."
        ),
        "engineering": (
            "Summarize for a senior engineer. 3 bullet points. "
            "Focus on technical details, system impact, and root cause."
        ),
        "customer": (
            "Summarize for a customer-facing communication. 2-3 sentences. "
            "Friendly, non-technical language. Focus on what happened and what we're doing about it."
        ),
    }
    system = audience_prompts.get(audience, audience_prompts["executive"])
    payload = build_chat_payload(system, text, max_tokens=300)
    return extract_content(call_chat_api(payload, api_key))

incident_text = (
    "At 14:32 UTC, the primary database cluster experienced a failover due to an expired "
    "TLS certificate. The failover took 45 minutes because health checks were misconfigured. "
    "12,000 customers experienced errors. Revenue impact estimated at $18,000. "
    "Root cause: certificate renewal was not in the automated rotation schedule. "
    "Fix: added certificate to rotation; health check config updated and tested."
)
for audience in ["executive", "engineering", "customer"]:
    print(f"\n--- {audience.upper()} ---")
    print(summarize_for_audience(incident_text, audience))

### Pattern 2 — Few-shot examples for consistent formatting

**Use case:** You want the model to classify tickets into exactly 5 categories with a specific output format. Providing 2–3 examples in the prompt dramatically improves consistency — the model follows the pattern.

**Pattern:** Include example input/output pairs in the system or user message.

In [ ]:
def classify_ticket_with_examples(ticket_text: str, api_key: str = None) -> str:
    """Classify a support ticket using few-shot examples for consistent output."""
    system = (
        "Classify the support ticket into exactly one category: "
        "billing, technical, account, feature_request, or general. "
        "Output format: CATEGORY: <category>\nCONFIDENCE: <high|medium|low>\nREASON: <one sentence>\n\n"
        "Examples:\n"
        "Ticket: 'My invoice shows a double charge for March.'\n"
        "CATEGORY: billing\nCONFIDENCE: high\nREASON: Customer reports incorrect charge on invoice.\n\n"
        "Ticket: 'The export button gives a 500 error.'\n"
        "CATEGORY: technical\nCONFIDENCE: high\nREASON: User reports server error on a specific feature.\n\n"
        "Ticket: 'Can we get dark mode in the dashboard?'\n"
        "CATEGORY: feature_request\nCONFIDENCE: high\nREASON: User requests a new UI feature."
    )
    payload = build_chat_payload(system, f"Ticket: '{ticket_text}'", max_tokens=100, temperature=0.1)
    return extract_content(call_chat_api(payload, api_key))

# Test with different ticket types
test_tickets = [
    "I can't log in after changing my email address.",
    "Please add support for SAML SSO integration.",
    "Why was I charged twice this month?",
]
for t in test_tickets:
    print(f"Ticket: {t}")
    print(classify_ticket_with_examples(t))
    print()

### Pattern 3 — Strict output format instructions

**Use case:** Downstream systems (dashboards, databases, alerting) consume structured data. Tell the model exactly what JSON keys to return and use `response_format` to enforce valid JSON.

**Already demonstrated** in Scenario 6 (risk classification) and Scenario 7 (content moderation) above. The key principle: be explicit about every field name, type, and possible value in the system prompt.

In [ ]:
# Summary: three prompt engineering patterns

PROMPT_PATTERNS = {
    "audience_specific": "Same content, different system prompts per audience (exec, eng, customer)",
    "few_shot": "2-3 examples in the prompt for consistent output formatting",
    "strict_json": "Explicit field names in system prompt + response_format: json_object",
}
print("Prompt engineering patterns for production:")
for name, desc in PROMPT_PATTERNS.items():
    print(f"  {name}: {desc}")
print()
print("These patterns apply to ALL text scenarios (§3). Use them together for best results.")

## 7. Multimodal reasoning workflow (one pipeline)

**Scenario:** Voice question → STT → text → LLM (with optional image context) → text answer → TTS. This is the flow for the **Student Lab Pack Lab** (multimodal assistant).

```
  [Voice/audio]  →  Block 1 (STT)  →  [Text]  →  Block 2 (Chat)  →  [Text]  →  Block 3 (TTS)  →  [Audio]
       ↑                                  ↑                                  ↑
   speech_to_text()              chat via build_chat_payload()       text_to_speech()
                                  + call_chat_api()
```

**Production deployment:**
```
  ┌─────────────┐     ┌──────────┐     ┌──────────┐     ┌──────────┐
  │ Voice Upload │────▶│ STT API  │────▶│ Chat API │────▶│ TTS API  │
  │ (frontend)   │     │ Block 1  │     │ Block 2  │     │ Block 3  │
  └─────────────┘     └──────────┘     └──────────┘     └──────────┘
        ↑                                                      │
        └──────────── Audio response ◀────────────────────────┘
```

**Next:** In **03-Lab-Multimodal-Assistant.ipynb** you will implement each block end-to-end.

### Implementing the pipeline in code

Below: one function that chains STT (if audio provided), Chat, and TTS (if output path provided). Text-only mode works without audio files — useful for testing and for users who type instead of speak.

In [ ]:
import time

def multimodal_assistant_pipeline(
    audio_path: str | None = None,
    text_query: str | None = None,
    image_url: str | None = None,
    tts_output_path: str | None = None,
    system_prompt: str = "You are a helpful assistant. Answer concisely.",
    api_key: str = None,
) -> dict:
    """Full pipeline: (audio OR text) → optional image context → Chat → optional TTS.
    Returns {query_text, answer_text, tts_path, timings}."""
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    timings: dict[str, float] = {}
    if not api_key:
        return {"query_text": None, "answer_text": "[Set OPENAI_API_KEY]", "tts_path": None, "timings": {}}

    # Block 1: STT (if audio provided)
    if audio_path and Path(audio_path).exists():
        t0 = time.time()
        query_text = speech_to_text(audio_path, api_key=api_key)
        timings["stt_seconds"] = round(time.time() - t0, 2)
        if not query_text:
            return {"query_text": None, "answer_text": "[STT failed]", "tts_path": None, "timings": timings}
    elif text_query:
        query_text = text_query
    else:
        return {"query_text": None, "answer_text": "[Provide audio_path or text_query]", "tts_path": None, "timings": {}}

    # Block 2: Chat (with optional image context)
    t0 = time.time()
    if image_url:
        # Vision-enhanced chat: text query + image
        base = CONFIG.get("openai_api_base", "https://api.openai.com/v1").rstrip("/")
        url = f"{base}/chat/completions"
        headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
        body = {
            "model": CONFIG.get("model_chat", "gpt-4o-mini"),
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": [
                    {"type": "text", "text": query_text},
                    {"type": "image_url", "image_url": {"url": image_url}},
                ]},
            ],
            "max_tokens": 300,
        }
        resp = requests.post(url, json=body, headers=headers, timeout=60)
        resp.raise_for_status()
        answer_text = resp.json()["choices"][0]["message"]["content"]
    else:
        payload = build_chat_payload(system_prompt, query_text, max_tokens=300)
        answer_text = extract_content(call_chat_api(payload, api_key))
    timings["chat_seconds"] = round(time.time() - t0, 2)

    # Block 3: TTS (optional)
    tts_path = None
    if tts_output_path and answer_text:
        t0 = time.time()
        if text_to_speech(answer_text, tts_output_path, api_key=api_key):
            tts_path = tts_output_path
        timings["tts_seconds"] = round(time.time() - t0, 2)

    return {"query_text": query_text, "answer_text": answer_text, "tts_path": tts_path, "timings": timings}

In [ ]:
# Run pipeline with text only (no audio file needed) — real-life: testing or text-only mode
result = multimodal_assistant_pipeline(text_query="What are three best practices for API security?")
print("Query:", result["query_text"])
print("Answer:", result["answer_text"])
print("TTS saved:", result["tts_path"])
print("Timings:", result["timings"])

In [ ]:
# With TTS: save the spoken reply to a file
# Uncomment to run (requires API key):
# result_tts = multimodal_assistant_pipeline(
#     text_query="Summarize the benefits of multimodal AI assistants in one sentence.",
#     tts_output_path="reply.mp3",
# )
# print("TTS file:", result_tts["tts_path"])
# print("Timings:", result_tts["timings"])
print("Pipeline with TTS: uncomment above and run with API key to generate reply.mp3")

### Production pattern — Per-block error recovery

**Use case:** In a deployed assistant, if TTS fails (e.g. service outage), the pipeline should still return the text answer. If STT fails, fall back to text input. Never lose the entire response because one optional block fails.

In [ ]:
def resilient_pipeline(
    audio_path: str | None = None,
    text_query: str | None = None,
    tts_output_path: str | None = None,
    api_key: str = None,
) -> dict:
    """Pipeline with per-block try/except. Degrades gracefully on partial failure."""
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
    result = {"query_text": None, "answer_text": None, "tts_path": None, "errors": []}
    if not api_key:
        result["errors"].append("No API key")
        return result

    # Block 1: STT (optional; fall back to text_query)
    query_text = text_query
    if audio_path and Path(audio_path).exists():
        try:
            query_text = speech_to_text(audio_path, api_key=api_key)
        except Exception as e:
            result["errors"].append(f"STT failed: {e}")
            # Fall back to text_query if available
            if not text_query:
                result["errors"].append("No text fallback available")
                return result
            query_text = text_query
    if not query_text:
        result["errors"].append("No input (audio or text)")
        return result
    result["query_text"] = query_text

    # Block 2: Chat (required)
    try:
        payload = build_chat_payload("You are a helpful assistant.", query_text, max_tokens=300)
        result["answer_text"] = extract_content(call_chat_api(payload, api_key))
    except Exception as e:
        result["errors"].append(f"Chat failed: {e}")
        return result

    # Block 3: TTS (optional; skip on failure)
    if tts_output_path and result["answer_text"]:
        try:
            text_to_speech(result["answer_text"], tts_output_path, api_key=api_key)
            result["tts_path"] = tts_output_path
        except Exception as e:
            result["errors"].append(f"TTS failed (text answer still available): {e}")

    return result

# Test resilient pipeline (text-only, no TTS)
r = resilient_pipeline(text_query="What is the capital of Japan?")
print("Answer:", r["answer_text"])
print("Errors:", r["errors"] if r["errors"] else "None")

### Production pattern — Pipeline telemetry

**Use case:** Track how long each block takes (STT: 2.3s, Chat: 1.1s, TTS: 0.8s) to identify bottlenecks. Log telemetry for monitoring dashboards.

In [ ]:
import logging

logger = logging.getLogger("multimodal_pipeline")

def pipeline_with_telemetry(text_query: str, api_key: str = None) -> dict:
    """Pipeline with structured telemetry logging per block."""
    import uuid
    request_id = str(uuid.uuid4())[:8]
    api_key = api_key or os.environ.get("OPENAI_API_KEY", "")

    telemetry = {"request_id": request_id, "blocks": {}}

    # Block 2: Chat (text-only for this demo)
    t0 = time.time()
    payload = build_chat_payload("You are a helpful assistant.", text_query, max_tokens=200)
    resp = call_chat_api(payload, api_key)
    content, meta = parse_chat_response(resp)
    elapsed = round(time.time() - t0, 3)

    telemetry["blocks"]["chat"] = {
        "elapsed_seconds": elapsed,
        "model": meta.get("model"),
        "prompt_tokens": meta.get("prompt_tokens"),
        "completion_tokens": meta.get("completion_tokens"),
        "estimated_cost_usd": round(estimate_cost(meta), 6),
    }
    telemetry["total_seconds"] = elapsed
    telemetry["answer"] = content

    logger.info("Pipeline telemetry: %s", json.dumps(telemetry, indent=2))
    return telemetry

tel = pipeline_with_telemetry("Name three benefits of REST APIs.")
print(f"Request {tel['request_id']}:")
print(f"  Chat: {tel['blocks']['chat']['elapsed_seconds']}s, "
      f"tokens: {tel['blocks']['chat'].get('prompt_tokens', '?')}+{tel['blocks']['chat'].get('completion_tokens', '?')}, "
      f"cost: ${tel['blocks']['chat'].get('estimated_cost_usd', 0):.6f}")
print(f"  Answer: {tel['answer'][:150]}")

## Summary

- **Text (primary focus):** Summarization, support drafts, compliance extraction, risk classification, content moderation, structured JSON output, multi-turn conversations, chain-of-thought reasoning — all via `chat/completions` with different system prompts.
- **Image (focused):** Vision API for screenshot description, OCR, invoice extraction, and combined image+text diagnosis — same JSON pattern with `image_url` in the content array.
- **Speech:** STT (multipart POST → text) and TTS (JSON POST → binary audio) — voice assistants, meeting transcripts, batch processing.
- **Multimodal workflow:** Chain APIs (STT → Chat → TTS) with per-block telemetry, error recovery, and graceful degradation.
- **Production patterns:** Response metadata/cost tracking, model fallback, prompt versioning, output post-processing, token estimation, batch processing with skip-failures.
- **Prompt engineering:** Audience-specific prompts, few-shot examples, strict JSON output — cross-cutting techniques for all scenarios.

**Next:** In **03-Lab-Multimodal-Assistant.ipynb**, implement the full voice → STT → Chat → TTS pipeline end-to-end, mapping each step to the blocks above.